## Single input Layer logisitic regression using Neural Net

In [181]:
import torch

In [182]:
tensor0d = torch.tensor(1.0)
tensor1d = torch.tensor([1,2,3])
tensor2d = torch.tensor([[1,2,3],[1,2,4]])
tensor3d = torch.Tensor([[[1,2],[4,3]],[[5,6],[7,8]]])


In [183]:
print(tensor0d.dtype)
print(tensor3d.shape)
print(tensor2d.reshape(3,2))

torch.float32
torch.Size([2, 2, 2])
tensor([[1, 2],
        [3, 1],
        [2, 4]])


In [184]:
print(tensor2d @ tensor2d.T)
print(tensor2d.matmul(tensor2d.T))

tensor([[14, 17],
        [17, 21]])
tensor([[14, 17],
        [17, 21]])


In [185]:
import torch.nn.functional as F
from torch.autograd import grad
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad= True)
b = torch.tensor([0.0], requires_grad= True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a,y)
grad_L_w1 = grad(loss,w1, retain_graph = True)
grad_L_b = grad(loss,b, retain_graph = True)

In [186]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [187]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## Multilayer neural network

In [188]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs,30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30,20),
            torch.nn.ReLU(),

            #output layer
            torch.nn.Linear(20,num_outputs),
        )
    def forward(self,x):
        logits = self.layers(x)
        return logits
        
    

In [236]:
model = NeuralNetwork(2,2)

In [220]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [237]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 752


In [223]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0281,  0.1284,  0.1009,  ..., -0.0914, -0.1354,  0.1046],
        [ 0.1062, -0.0360,  0.0624,  ...,  0.0697, -0.0964,  0.0765],
        [-0.0045,  0.0584, -0.0003,  ..., -0.1287, -0.0371, -0.1072],
        ...,
        [-0.1057,  0.1387, -0.1316,  ...,  0.0019, -0.0163,  0.0589],
        [-0.1053, -0.0406, -0.1302,  ...,  0.0139, -0.0647, -0.0738],
        [-0.0511,  0.0880,  0.0709,  ...,  0.1037,  0.0555,  0.0037]],
       requires_grad=True)


In [193]:
torch.manual_seed(123)
X = torch.rand((1, 51))
out = model(X)
print(out)

tensor([[0.0700, 0.2537, 0.2538]], grad_fn=<AddmmBackward0>)


In [194]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.2938, 0.3531, 0.3531]])


## Data Loader

In [195]:
X_train = torch.tensor([
[-1.2, 3.1],
[-0.9, 2.9],
[-0.5, 2.6],
[2.3,-1.1],
[2.7,-1.5]])
y_train = torch.tensor([0, 0, 0, 1, 1])
X__test = torch.tensor([
[-0.8, 2.8],
[2.6,-1.6],])
y__test = torch.tensor([0, 1])

In [196]:
from torch.utils.data import Dataset,DataLoader 

class ToyDataset(Dataset):
    def __init__(self, X, y):
            self.features = X
            self.labels = y
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    def __len__(self):
        return self.labels.shape[0]


train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X__test, y__test)

In [197]:
print(len(train_ds))

5


In [224]:
torch.manual_seed(123)

train_loader = DataLoader(dataset=train_ds, batch_size=2, shuffle=True, num_workers=0,drop_last=True)
test_loader = DataLoader(dataset=test_ds, batch_size=2, shuffle=False, num_workers=0)

In [233]:
for idx,(x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:")
    print("Features:", x)
    print("Labels:", y)
    print()

Batch 1:
Features: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]])
Labels: tensor([0, 0])

Batch 2:
Features: tensor([[-0.9000,  2.9000],
        [ 2.7000, -1.5000]])
Labels: tensor([0, 1])



In [235]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features,labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch: {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Loss: {loss.item():.4f}")
    model.eval()

Epoch: 001/003 | Batch: 001/002 | Loss: 0.7487
Epoch: 001/003 | Batch: 002/002 | Loss: 0.6450
Epoch: 002/003 | Batch: 001/002 | Loss: 0.4423
Epoch: 002/003 | Batch: 002/002 | Loss: 0.1256
Epoch: 003/003 | Batch: 001/002 | Loss: 0.0269
Epoch: 003/003 | Batch: 002/002 | Loss: 0.0043


In [218]:
with torch.no_grad():
    out = model(X_train)
print(out)

tensor([[ 0.8577, -1.5620],
        [ 0.7605, -1.4302],
        [ 0.6258, -1.2453],
        [-0.2900,  0.2092],
        [-0.3425,  0.2564]])


In [202]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(out, dim=1)
print(probas)

tensor([[0.9183, 0.0817],
        [0.8994, 0.1006],
        [0.8666, 0.1334],
        [0.3777, 0.6223],
        [0.3546, 0.6454]])


In [174]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device) 
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()    
    print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"f" | Batch {batch_idx:03d}/{len(train_loader):03d}" f" | Train/Val Loss: {loss:.2f}")
model.eval()

Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)

In [240]:
def compute_accuracy(model, dataloader):
    model = model.eval()
    correct = 0.0
    total_examples = 0
    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions #A
        correct += torch.sum(compare) #B
    total_examples += len(compare)
    return (correct / total_examples).item() #C

In [241]:
print(compute_accuracy(model, train_loader))
print(compute_accuracy(model, test_loader))

1.5
0.5


### Time checking between CPU vs GPU

In [175]:
import torch

n = 1000

a_cpu = torch.rand((n, n))
b_cpu = torch.rand((n, n))

In [176]:
%timeit a_cpu @ b_cpu

1.28 ms ± 9.68 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [177]:
device = torch.device("mps")

a_gpu = torch.rand((n, n), device=device)
b_gpu = torch.rand((n, n), device=device)

%timeit a_gpu @ b_gpu

755 μs ± 5.88 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
